In [ ]:
RENDER_ENV = False
LOAD_MODEL = False
new_size = (96,136) #(84,84)
batch_size = 32
num_episodes = 4680 #56160 12 horas #4680 para 1 hora pc Seba
max_episode_steps = 100
num_stacked_frames = 1
intervals = 4
Model = "DQN" # DQN o PPO
version = 6

# install y imports

In [ ]:
!pip install tetris_gymnasium
!pip install stable_baselines3

In [ ]:
file_path = '/usr/local/lib/python3.12/dist-packages/tetris_gymnasium/wrappers/observation.py'

with open(file_path, 'r') as f:
    content = f.read()
modified_content = content.replace("high=len(env.unwrapped.tetrominoes)", "high=255")
modified_content = modified_content.replace("self.render_scaling_factor", "20")

with open(file_path, 'w') as f:
    f.write(modified_content)

with open(file_path, 'r') as f:
    print(f.read())


In [ ]:
file_path = '/usr/local/lib/python3.12/dist-packages/tetris_gymnasium/wrappers/grouped.py'

with open(file_path, 'r') as f:
    content = f.read()
modified_content = content.replace("high=env.unwrapped.height * env.unwrapped.width", "high=255")
modified_content = modified_content.replace("self.render_scaling_factor", "20")
modified_content = modified_content.replace("dtype=np.float32", "dtype=np.uint8")

with open(file_path, 'w') as f:
    f.write(modified_content)

with open(file_path, 'r') as f:
    print(f.read())


In [ ]:
import gymnasium as gym
import numpy as np
from tetris_gymnasium.envs.tetris import Tetris
from tetris_gymnasium.wrappers.observation import RgbObservation, FeatureVectorObservation
from tetris_gymnasium.wrappers.grouped import GroupedActionsObservations
from gymnasium.wrappers import TimeLimit, ResizeObservation, RecordVideo, FrameStackObservation, GrayscaleObservation
from stable_baselines3 import DQN, PPO
import os
from stable_baselines3.common.buffers import ReplayBuffer

# Funciones

In [ ]:
def get_last_modified_file(directory_path):
    if not os.path.isdir(directory_path):
        print(f"Error: Directory '{directory_path}' does not exist.")
        return None
    files = [os.path.join(directory_path, f) for f in os.listdir(directory_path) if os.path.isfile(os.path.join(directory_path, f))]
    if not files:
        return None
    files.sort(key=os.path.getmtime, reverse=True)
    return files[0]

target_directory = f"../Models_Saves/{Model}"  # Replace with your directory path
model_load_path = get_last_modified_file(target_directory)

if model_load_path:
    print(f"The last modified file is: {model_load_path}")
else:
    print("No files found in the directory or directory does not exist.")

In [ ]:
try:
    os.mkdir("../Models_Saves")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Models_Saves/PPO")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Models_Saves/DQN")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA/PPO")
except Exception as e:
    print(f"Error: {e}")
try:
    os.mkdir("../Video_Tetris_IA/DQN")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
def calc_max_height(mat):
  mat1 = np.rot90(np.rot90(mat))
  for i in range(len(mat1)):
    count = 0
    for col in mat1[i]:
      if col > 0:
        count+=1
    if count == 0:
      return i-1
  return 19

def calc_holes(mat,height):
  mat1 = np.rot90(np.rot90(np.rot90(mat)))
  holes = 0
  for row in mat1:
    for i in range(height+1):
      if row[i] == 0:
        holes += 1
  return holes

def calc_adj_col(mat, height):
  mat1 = np.rot90(np.rot90(np.rot90(mat)))
  prev_height = -1
  dif_count = 0
  for row in mat1:
    act_height = 0
    for i in range(height+1):
      if row[i] != 0:
        act_height = i+1
    if prev_height >= 0:
      dif_count+=(abs(prev_height-act_height))
    prev_height = act_height
  return dif_count


In [ ]:

class CustomRewardWrapper(gym.RewardWrapper):
    #def __init__(self, env, holes_penalty = 0.005, height_penalty = 0.05, increase_height_penalty = 0.4, increase_holes_penalty = 0.2, dif_heigh_penalty = 0.03):
    #Prometedor pero algo falta def __init__(self, env, holes_penalty = 0.000002, height_penalty = 0.00002, increase_height_penalty = 0.4, increase_holes_penalty = 0.2, dif_heigh_penalty = 0.00001):
    def __init__(self, env, holes_penalty = 0.000002, height_penalty = 0.00002, increase_height_penalty = 0.2, increase_holes_penalty = 0.4, dif_heigh_penalty = 0.00001):
        super(CustomRewardWrapper, self).__init__(env)
        self.holes_penalty = holes_penalty
        self.height_penalty = height_penalty
        self.increase_height_penalty = increase_height_penalty
        self.increase_holes_penalty = increase_holes_penalty
        self.dif_heigh_penalty = dif_heigh_penalty

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)

        game_variables = self.env.unwrapped.get_state().board[:-4, 4:-4]
        self.previous_max_height = calc_max_height(game_variables)
        self.previous_holes = calc_holes(game_variables, self.previous_max_height)

        return obs, info

    def reward(self, reward):
        #print(f"Reward original: {reward}")
        # Probar mayor penalizacion de agujeros
        custom_reward = reward
        game_variables = self.env.unwrapped.get_state().board[:-4, 4:-4]

        if game_variables.any():
            # Calcula la altura maxima actual
            current_max_height = calc_max_height(game_variables)
            # Calcula ls espacios vacios entre la base y la altura
            current_holes = calc_holes(game_variables, current_max_height)
            #Calcula la diferencia de altura entre las columnas adyacentes y las suma
            current_height_dif = calc_adj_col(game_variables, current_max_height)
            #Penalizacion constante por altura
            custom_reward -= current_max_height*self.height_penalty
            #Penalizacion constante por agujeros
            custom_reward -= current_holes*self.holes_penalty
            #Penalizacion constante por diferencia de altura entre columnas
            custom_reward -= current_height_dif*self.dif_heigh_penalty
            #Penalizacion y recompensa por aumentar altura o disminuir altura respectivamente
            if current_max_height-self.previous_max_height != 0:
                custom_reward -= (current_max_height-self.previous_max_height)*self.increase_height_penalty
            #Penalizacion y recompensa por aumentar agujeros o disminuir agujeros respectivamente
            if current_holes-self.previous_holes != 0:
                custom_reward -= (current_holes-self.previous_holes)*self.increase_holes_penalty
            #Actualiza valores previos para el siguiente paso
            self.previous_holes = current_holes
            self.previous_max_height = current_max_height
        return custom_reward

In [ ]:
def make_env(*, game, max_episode_steps=4500, obs_type, **kwargs):
    env = gym.make(game, **kwargs)
    if obs_type == "RGB":
      env = RgbObservation(env)
      env = ResizeObservation(env, new_size)
      env = GrayscaleObservation(env)
      env = FrameStackObservation(env, stack_size=num_stacked_frames)
    elif obs_type == "FV":
      env = FeatureVectorObservation(env)
    elif obs_type == "GA":
      env = GroupedActionsObservations(env)
      env = ResizeObservation(env, new_size)
    env = CustomRewardWrapper(env)
    env.reset(seed=42)
    return env

#Training

In [ ]:
try:
  env.close()
except:
  print('no hay env para cerrar')

In [ ]:
if __name__ == "__main__":
    env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array", obs_type="RGB")
    if Model == "DQN":
        model = DQN("CnnPolicy", env, buffer_size=20000, verbose=1, exploration_fraction=0.3, learning_rate=2e-5, batch_size=256, gamma=0.999) # para Mlp usar FeatureVectorObservation para Cnn usar RgbObservation
    else:
        model = PPO("CnnPolicy", env, verbose=1, n_steps=4096, clip_range=0.1, ent_coef=0.01, gamma=0.999, batch_size=256, learning_rate=2e-5) # para Mlp usar FeatureVectorObservation para Cnn

    if LOAD_MODEL:
        model.load(model_load_path)
    model.learn(total_timesteps=num_episodes*max_episode_steps, log_interval=4)
    model.save(f"../Models_Saves/{Model}/{Model}_V{version}_S{max_episode_steps*num_episodes}")
    env.close()

# Recording

In [ ]:
try:
  env.close()
except:
  print('no hay env para cerrar')

In [ ]:
env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array", obs_type="GA")
try:
  env = RecordVideo(
    env,
    video_folder=f'../Video_Tetris_IA/{Model}',    # Folder to save videos
    name_prefix=f'{Model}_eval-V{version}-Trained_steps_{num_episodes*max_episode_steps}',               # Prefix for video filenames
    episode_trigger=lambda x: True    # Record every episode
  )
except Exception as e:
  print(f'error implementando grabacion: {e}')
if Model == "DQN":
    model = DQN.load(f"../Models_Saves/{Model}/{Model}_V{version}_S{max_episode_steps*num_episodes}", env=env, print_system_info=True)
else:
    model = PPO.load(f"../Models_Saves/{Model}/{Model}_V{version}_S{max_episode_steps*num_episodes}", env=env, print_system_info=True)

In [ ]:
for episode in range(10):
  state, info = env.reset()
  total_reward = 0
  done = False
  step_count = 0
  while not done:
    step_count+=1
    action, _states = model.predict(state, deterministic=True)
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_reward += reward
  print(f"Episode: {episode} Reward: {total_reward} Steps: {step_count}")

In [ ]:
# var = len(env.unwrapped.get_state().board)
# count = var
# temp = env.unwrapped.get_state().board[:-4, 4:-4]
# print(temp)
# print(calc_max_height(temp))
# print(calc_holes(temp))
# #print(len(env.unwrapped.get_state().board))

In [ ]:
# from stable_baselines3.common.env_checker import check_env
# check_env(env)

# Testing Elements

In [ ]:
env.unwrapped.get_state().board[:-4, 4:-4]

In [ ]:
np.rot90(np.rot90(np.rot90(env.unwrapped.get_state().board[:-4, 4:-4])))

In [ ]:
height_test = calc_max_height(env.unwrapped.get_state().board[:-4, 4:-4])
print(height_test)
print("-"*20)
print(calc_adj_col(env.unwrapped.get_state().board[:-4, 4:-4], height_test))
print("-"*20)
print(height_test)
print(calc_holes(env.unwrapped.get_state().board[:-4, 4:-4], height_test))

In [ ]:
env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array")


In [ ]:
env.unwrapped.queue.get_queue()

# Testing Grouped

In [ ]:
try:
  env.close()
except:
  print('no hay env para cerrar')

In [ ]:
if __name__ == "__main__":
    env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array", obs_type="GA")
    if Model == "DQN":
        model = DQN("CnnPolicy", env, buffer_size=20000, verbose=1, exploration_fraction=0.3, learning_rate=2e-5, batch_size=256, gamma=0.999) # para Mlp usar FeatureVectorObservation para Cnn usar RgbObservation
    else:
        model = PPO("CnnPolicy", env, verbose=1, n_steps=4096, clip_range=0.1, ent_coef=0.01, gamma=0.999, batch_size=256, learning_rate=2e-5) # para Mlp usar FeatureVectorObservation para Cnn

    if LOAD_MODEL:
        model.load(model_load_path)
    model.learn(total_timesteps=num_episodes*max_episode_steps, log_interval=4)
    model.save(f"../Models_Saves/{Model}/{Model}_V{version}_S{max_episode_steps*num_episodes}")
    env.close()

# Testing Feature Vector


In [ ]:
try:
  env.close()
except:
  print('no hay env para cerrar')

In [ ]:
if __name__ == "__main__":
    env = make_env(game = "tetris_gymnasium/Tetris", render_mode="rgb_array", obs_type="FV")
    if Model == "DQN":
        model = DQN("MlpPolicy", env, buffer_size=20000, verbose=1, exploration_fraction=0.3, learning_rate=2e-5, batch_size=256, gamma=0.999) # para Mlp usar FeatureVectorObservation para Cnn usar RgbObservation
    else:
        model = PPO("MlpPolicy", env, verbose=1, n_steps=4096, clip_range=0.1, ent_coef=0.01, gamma=0.999, batch_size=256, learning_rate=2e-5) # para Mlp usar FeatureVectorObservation para Cnn

    if LOAD_MODEL:
        model.load(model_load_path)
    model.learn(total_timesteps=num_episodes*max_episode_steps, log_interval=4)
    model.save(f"../Models_Saves/{Model}/{Model}_V{version}_S{max_episode_steps*num_episodes}")
    env.close()